# Sinistros de Trânsito e Infraestrutura Semafórica em Fortaleza
**Aluno:** Yuri Lucas Luz da Silva — Matrícula: 2223481

In [80]:
import warnings; warnings.filterwarnings('ignore')
import os, requests, json
import geopandas as gpd
import pandas as pd

os.makedirs('data', exist_ok=True)

URLS = {
    'data/sinistros.geojson': 'https://dados.fortaleza.ce.gov.br/dataset/76c32db6-97b4-439a-9f12-c0f6413b2013/resource/75f7e594-cab6-4907-b682-9b6af11ebc2c/download/sinistros-2015-2024.geojson',
    'data/semaforos.geojson': 'https://dados.fortaleza.ce.gov.br/dataset/c529ba45-27ae-4d3a-8f70-76019a87edba/resource/4e4a18a1-86cc-48a1-b631-3ef636b455ee/download/dadosabertos_semaforosctafor.geojson',
}

for path, url in URLS.items():
    if not os.path.exists(path):
        print(f'Baixando {os.path.basename(path)}...')
        r = requests.get(url, verify=False, stream=True, timeout=120)
        with open(path, 'wb') as f:
            for chunk in r.iter_content(256*1024): f.write(chunk)
        print(f'  {os.path.getsize(path)/1e6:.1f} MB')
    else:
        print(f'OK  {os.path.basename(path)}')

OK  sinistros.geojson
OK  semaforos.geojson


---
## 1. Sinistros de Trânsito (2015–2024)

In [81]:
sinistros = gpd.read_file('data/sinistros.geojson')

# Aparentemente, a coluna HORA não é exportada pelo 
# geopandas (Skipping field HORA: unsupported OGR type: 10), 
# então vamos extrair diretamente do GeoJSON e adicionar ao GeoDataFrame
with open('data/sinistros.geojson') as f:
    feats = json.load(f)['features']
sinistros['HORA'] = [ft['properties'].get('HORA') for ft in feats]

print(f'Registros : {len(sinistros):,}')
print(f'Colunas   : {len(sinistros.columns)}')
print(f'Período   : {sinistros.ANO.min()} – {sinistros.ANO.max()}')

Skipping field HORA: unsupported OGR type: 10


Registros : 130,823
Colunas   : 28
Período   : 2015 – 2024


In [82]:
# campos e exemplos de valores.
# Removi a coluna geometry para não poluir a visualização, mas ela está presente no GeoDataFrame.
sample = sinistros.drop(columns='geometry').sample(1).squeeze()
sample_table = pd.DataFrame({
    'valor': sample,
    '%_ausentes': sinistros.drop(columns='geometry').isnull().mean().mul(100).round(1),
    'únicos': sinistros.drop(columns='geometry').nunique(),
    'tipo': sinistros.drop(columns='geometry').dtypes,
})
sample_table.reset_index().rename(columns={'index': 'coluna'})

,coluna,valor,%_ausentes,únicos,tipo
0,COD_ACIDENTE,342198,0.0,130820,object
1,DIA,28,0.0,31,int32
2,MES,2,0.0,12,int32
3,ANO,2019,0.0,10,int32
4,LOG1,"RUA AMERICO ROCHA LIMA,",0.0,41243,object
5,LOG2,"RUA FERNANDO FARIAS DE MELO (MANUEL SÁTIRO),",0.0,7793,object
6,NUMERO,NaN,50.0,6225,float64
7,LATITUDE,-3.79908,0.0,72066,float64
8,LONGITUDE,-38.578449,0.0,71201,float64
9,TIPOCRUZAMENTO,Cruz,0.0,14,object


In [83]:
# valores únicos das colunas categóricas relevantes
for col in ['SEVERIDADE', 'NATUREZA', 'TIPOCRUZAMENTO', 'CONTROLETRAFEGO', 'USOSOLO']:
    print(f'{col}: {sorted(sinistros[col].dropna().unique().tolist())}')

SEVERIDADE: ['Fatal', 'Ferido', 'Ileso', 'Não informado']
NATUREZA: ['ACIDENTE ONIBUS (TRANSPORTE COLETIVO)', 'Acidente Pessoal', 'Acidente com transporte ferroviário', 'Atropelamento', 'Atropelamento de Animal', 'Capotamento', 'Choque', 'Choque c/ Obstáculo Fixo', 'Colisão', 'Colisão Frontal', 'Colisão Lateral', 'Colisão Lateral Sentido Oposto', 'Colisão Transversal', 'Colisão Traseira', 'Engavetamento', 'Não informado', 'Outros', 'Queda', 'Queda de moto', 'Tombamento']
TIPOCRUZAMENTO: ['Com Via Férrea', 'Com via férrea', 'Cruz', 'Cruzamento', 'Duplo T', 'Meio De Quadra', 'Meio de Quadra', 'Meio de quadra', 'Não Informado', 'Não informado', 'Outros', 'Rotatória', 'T', 'Y']
CONTROLETRAFEGO: ['Cancela', 'Cancela Danificada', 'Dê Preferência', 'Faixa De Pedestre', 'Faixa de pedestre', 'Gesto', 'Não Há Controle', 'Não Informado', 'Não há Controle', 'Não informado', 'Outros', 'Pare', 'Semáforo', 'Semáforo Danificado', 'Semáforo Intermitente', 'Semáforo Manual']
USOSOLO: ['Cruzamento', 'Lon

---
## 1.1. Parque Semafórico

In [84]:
semaforos = gpd.read_file('data/semaforos.geojson')

print(f'Registros : {len(semaforos):,}')
print(f'Colunas   : {len(semaforos.columns)}')

Registros : 1,231
Colunas   : 26


In [85]:
# campos e exemplos de valores.
# Removi a coluna geometry para não poluir a visualização, mas ela está presente no GeoDataFrame.
sample_s = semaforos.drop(columns='geometry').sample(1).squeeze()
sample_table_s = pd.DataFrame({
    'valor': sample_s,
    '%_ausentes': semaforos.drop(columns='geometry').isnull().mean().mul(100).round(1),
    'únicos': semaforos.drop(columns='geometry').nunique(),
    'tipo': semaforos.drop(columns='geometry').dtypes,
})
sample_table_s.reset_index().rename(columns={'index': 'coluna'})

,coluna,valor,%_ausentes,únicos,tipo
0,CÓDIGO,649,0.0,1225,object
1,CRUZAMENTO,R. CEL. JUCÁ & R. BENI CARVALHO,0.0,1217,object
2,STATUS,CENTRALIZADO,0.0,5,object
3,DATA_IMPLANTAÇÃO,06/07/2011,30.5,707,object
4,DATA_DESATIVAÇÃO,None,88.3,120,object
5,DATA_REATIVAÇÃO,None,95.4,45,object
6,GRUPO,SUB 08,11.5,180,object
7,MODO_CONTROLE,SCOOT CONJUGADO,11.5,6,object
8,QUANT_GF_T,2.0,11.6,10,float64
9,QUANT_GF_I,1.0,11.6,7,float64


In [86]:
# valores únicos das colunas de configuração
col_est = next((c for c in semaforos.columns if 'EST' in c.upper() and 'GIO' in c.upper()), None)
for col in ['STATUS', 'MODO_CONTROLE'] + ([col_est] if col_est else []):
    print(f'{col}: {sorted(semaforos[col].dropna().unique().tolist())}')

STATUS: ['CENTRALIZADO', 'CONVENCIONAL', 'DESATIVADO', 'DETRAN', 'PROJETO']
MODO_CONTROLE: ['ECOTRAFIX', 'ECOTRAFIX CONJUGADO', 'LOCAL', 'LOCAL CONJUGADO', 'SCOOT', 'SCOOT CONJUGADO']
ESTÁGIOS: [2.0, 3.0, 4.0]


---
## 2. Integração com o Parque Semafórico e Limpeza dos Dados

A partir daqui (Parte 2) enriqueci os sinistros com o parque semafórico (fonte externa) e
preparei um dataset limpo. Cada bloco faz uma etapa: limpeza dos sinistros, limpeza dos semáforos,
reprojeção métrica, reconstrução temporal do parque, integração espacial e o dataset final.

In [87]:
import numpy as np

# Parâmetros da integração

# Os dois datasets possuem o mesmo padrão (SIRGAS 2000) que está descrito no próprio dataframe.
# Para medir distâncias em metros, precisamos reprojetar para um CRS métrico adequado para Fortaleza,
# ou seja, usando o UTM 24S que é o timezone local. O EPSG correspondente é 31984. 
CRS_METRICO = 31984           # SIRGAS 2000 / UTM 24S (metros) — adequado para Fortaleza
RAIO_INTERSECCAO_M = 30       # distância máx. para considerar "semáforo na interseção"
RAIOS_DENSIDADE = (100, 250)  # raios (m) para contar semáforos ao redor de cada sinistro

In [88]:
# Confirmação do CRS de origem, lido do próprio arquivo: EPSG:4674 = SIRGAS 2000 (em graus).
# (ainda NÃO reprojetado aqui — a reprojeção para 31984 acontece no bloco 2.3)
print('Sinistros ->', sinistros.crs.name, '| EPSG:', sinistros.crs.to_epsg())
print('Semáforos ->', semaforos.crs.name, '| EPSG:', semaforos.crs.to_epsg())

Sinistros -> SIRGAS 2000 | EPSG: 4674
Semáforos -> SIRGAS 2000 | EPSG: 4674


### 2.1 Limpeza dos sinistros

Padronizei as categóricas com variações de caixa/acento, montei a `data_hora` e removi a
coluna `NUMERO` (~50% ausente). Todas as linhas são mantidas.

In [89]:
# Normaliza espaços e padroniza variações de caixa/acento (variante -> rótulo único)
def normaliza_texto(serie):
    return serie.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)

MAPA_TIPOCRUZAMENTO = {
    'Com via férrea': 'Com Via Férrea',
    'Meio De Quadra': 'Meio de Quadra', 'Meio de quadra': 'Meio de Quadra',
    'Não Informado': 'Não informado',
}
MAPA_CONTROLETRAFEGO = {
    'Faixa De Pedestre': 'Faixa de Pedestre', 'Faixa de pedestre': 'Faixa de Pedestre',
    'Não há Controle': 'Não Há Controle', 'Não Informado': 'Não informado',
}
MAPA_NATUREZA = {'ACIDENTE ONIBUS (TRANSPORTE COLETIVO)': 'Acidente Ônibus (Transporte Coletivo)'}

sinistros['TIPOCRUZAMENTO']  = normaliza_texto(sinistros['TIPOCRUZAMENTO']).replace(MAPA_TIPOCRUZAMENTO)
sinistros['CONTROLETRAFEGO'] = normaliza_texto(sinistros['CONTROLETRAFEGO']).replace(MAPA_CONTROLETRAFEGO)
sinistros['NATUREZA']        = normaliza_texto(sinistros['NATUREZA']).replace(MAPA_NATUREZA)
sinistros['USOSOLO']         = normaliza_texto(sinistros['USOSOLO'])
sinistros['SEVERIDADE']      = normaliza_texto(sinistros['SEVERIDADE'])

# data_hora a partir de DIA/MES/ANO + HORA (datas impossíveis viram NaT)
base = pd.to_datetime(dict(year=sinistros['ANO'], month=sinistros['MES'], day=sinistros['DIA']), errors='coerce')
hora = pd.to_timedelta(sinistros['HORA'].fillna('00:00:00'), errors='coerce').fillna(pd.Timedelta(0))
sinistros['data_hora'] = base + hora

# NUMERO (número do imóvel): ~50% ausente e pouco útil -> remove
sinistros = sinistros.drop(columns=['NUMERO'])

print('Limpeza dos sinistros concluída |', f'{len(sinistros):,}', 'linhas')
print('Categorias em TIPOCRUZAMENTO após padronização:', sinistros['TIPOCRUZAMENTO'].nunique())

Limpeza dos sinistros concluída | 130,823 linhas
Categorias em TIPOCRUZAMENTO após padronização: 10


### 2.2 Limpeza dos semáforos

Converti as datas (formato DD/MM/AAAA), tratei `ESTÁGIOS` (mantendo ausentes) e padronizei
`STATUS`/`MODO_CONTROLE` (ausente vira "Desconhecido").

In [90]:
for col in ['DATA_IMPLANTAÇÃO', 'DATA_DESATIVAÇÃO', 'DATA_REATIVAÇÃO']:
    semaforos[col] = pd.to_datetime(semaforos[col], dayfirst=True, errors='coerce')

semaforos['estagios_num'] = semaforos['ESTÁGIOS'].astype('Int64')
semaforos['STATUS'] = normaliza_texto(semaforos['STATUS'])
semaforos['MODO_CONTROLE'] = normaliza_texto(semaforos['MODO_CONTROLE']).fillna('Desconhecido')

print('Limpeza dos semáforos concluída')
print('ESTÁGIOS ausentes:', int(semaforos['estagios_num'].isna().sum()))
print('STATUS:', dict(semaforos['STATUS'].value_counts()))

Limpeza dos semáforos concluída
ESTÁGIOS ausentes: 143
STATUS: {'CENTRALIZADO': np.int64(598), 'CONVENCIONAL': np.int64(491), 'DESATIVADO': np.int64(111), 'DETRAN': np.int64(25), 'PROJETO': np.int64(6)}


### 2.3 Reprojeção para coordenadas métricas

Para medir distâncias em metros, reprojetei as duas camadas para EPSG:31984 (UTM 24S).

In [91]:
sinistros = sinistros.to_crs(CRS_METRICO)
semaforos = semaforos.to_crs(CRS_METRICO)
print('CRS atual (deve ser 31984):', sinistros.crs.to_epsg())

CRS atual (deve ser 31984): 31984


### 2.4 Reconstrução do parque semafórico por ano

O dataset de semáforos reflete o parque **atual**, mas os sinistros vão de 2015 a 2024. Para cada
ano, considerei apenas os semáforos que provavelmente já existiam, usando as datas de
implantação/desativação/reativação e o `STATUS`.

**Pressupostos:** 
- implantação ausente = já existia
- desativação ausente = nunca desativado
- `STATUS = "PROJETO"` = nunca existiu (excluído) 
- `"DESATIVADO"` sem data = removido (inativo)

In [92]:
def semaforos_ativos_no_ano(semaforos, ano):
    # Subconjunto de semáforos provavelmente ativos no ano informado
    s = semaforos[semaforos['STATUS'] != 'PROJETO'].copy()
    implantacao = s['DATA_IMPLANTAÇÃO'].dt.year
    desativacao = s['DATA_DESATIVAÇÃO'].dt.year
    reativacao  = s['DATA_REATIVAÇÃO'].dt.year
    ja_existia          = implantacao.isna() | (implantacao <= ano)
    foi_desativado      = desativacao.notna() & (desativacao <= ano)
    foi_reativado       = reativacao.notna()  & (reativacao  <= ano)
    desativado_sem_data = (s['STATUS'] == 'DESATIVADO') & s['DATA_DESATIVAÇÃO'].isna()
    return s[ja_existia & ((~foi_desativado) | foi_reativado) & (~desativado_sem_data)]

# Aqui é possível entendermos que o número de semáforos ativos varia positivamente ao longo dos 
# anos, o que é esperado, já que a cidade tem expandido seu parque semafórico. Além disso
# deixa claro que foi uma boa decisão considerarmos a implantação/desativação/reativação para 
# reconstruir o parque semafórico por ano, em vez de assumir que todos os semáforos do dataset já 
# existiam desde o início (2015).
print('Semáforos ativos em 2015:', len(semaforos_ativos_no_ano(semaforos, 2015)))
print('Semáforos ativos em 2018:', len(semaforos_ativos_no_ano(semaforos, 2018)))
print('Semáforos ativos em 2022:', len(semaforos_ativos_no_ano(semaforos, 2022)))
print('Semáforos ativos em 2025:', len(semaforos_ativos_no_ano(semaforos, 2025)))

Semáforos ativos em 2015: 754
Semáforos ativos em 2018: 914
Semáforos ativos em 2022: 1081
Semáforos ativos em 2025: 1115


### 2.5 Integração espacial: distância, densidade e interseção

Para cada sinistro, no ano correspondente, calculei:
- `dist_semaforo_m` — distância ao semáforo ativo mais próximo (metros);
- atributos do semáforo mais próximo (`estágios`, `modo`, `status`);
- `tem_semaforo_interseccao` — 1 se o semáforo mais próximo está a ≤ 30 m.
- `densidade_semaforos_100m` / `_250m` — nº de semáforos ativos ao redor;

Obs: em `densidade_semaforos_x` os raios foram escolhidas com base nas dimensões das quadras de fortaleza:
1. 80m ~ 120m (tamanho típico de uma quadra) para capturar semáforos na mesma quadra;
2. 250m para capturar semáforos em quadras adjacentes, considerando a influência de semáforos 
próximos, mesmo que não estejam na mesma quadra.



In [93]:
COLS_SEMAFORO = ['geometry', 'CÓDIGO', 'STATUS', 'MODO_CONTROLE', 'estagios_num']
RENOMEIA = {'CÓDIGO': 'semaforo_proximo_codigo', 'STATUS': 'semaforo_proximo_status',
            'MODO_CONTROLE': 'semaforo_proximo_modo', 'estagios_num': 'semaforo_proximo_estagios'}

def integrar_ano(grupo, ativos):
    grupo = grupo.copy()
    # densidade: nº de semáforos ativos dentro de cada raio (índice espacial)
    indice = ativos.sindex
    pontos = grupo.geometry.values
    for raio in RAIOS_DENSIDADE:
        pares = indice.query(pontos, predicate='dwithin', distance=raio)
        grupo[f'densidade_semaforos_{raio}m'] = np.bincount(pares[0], minlength=len(grupo)).astype('int32')
        
    # semáforo mais próximo + atributos
    selecao = ativos[COLS_SEMAFORO].rename(columns=RENOMEIA)
    juncao = gpd.sjoin_nearest(grupo, selecao, how='left', distance_col='dist_semaforo_m')
    juncao = juncao[~juncao.index.duplicated(keep='first')]   # desempata vizinhos múltiplos
    return juncao.drop(columns='index_right', errors='ignore')

partes = []
for ano, grupo in sinistros.groupby('ANO', sort=True):
    ativos = semaforos_ativos_no_ano(semaforos, ano)
    partes.append(integrar_ano(grupo, ativos))
sinistros = pd.concat(partes).sort_index()

sinistros['tem_semaforo_interseccao'] = (sinistros['dist_semaforo_m'] <= RAIO_INTERSECCAO_M).astype('int8')
print('Integração concluída |', f'{len(sinistros):,}', 'sinistros enriquecidos')
print(sinistros['dist_semaforo_m'].describe().round(1))

Integração concluída | 130,823 sinistros enriquecidos
count    130823.0
mean        268.9
std         402.7
min           0.0
25%          56.4
50%         147.0
75%         330.6
max        6675.3
Name: dist_semaforo_m, dtype: float64


### 2.6 Imputação de valores ausentes

Trato as ausências restantes com estratégias simples e documentadas. Para
`semaforo_proximo_estagios` (numérica, ~8% ausente) uso a **moda da rua** (`LOG1`) — como a mesma
via tende a ter semáforos de configuração parecida —, com *fallback* na moda global quando a rua
não tem nenhum valor conhecido. Guardo a flag `estagios_imputado` para separar, nas análises, os
valores reais dos imputados.

In [94]:
# Marca o que será imputado
sinistros['estagios_imputado'] = sinistros['semaforo_proximo_estagios'].isna()

# 1) Moda por rua (LOG1): a mesma via tende a ter semáforos de configuração parecida
moda_por_rua = (sinistros.groupby('LOG1')['semaforo_proximo_estagios']
                .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan))
estagios = sinistros['semaforo_proximo_estagios'].fillna(sinistros['LOG1'].map(moda_por_rua))

# 2) Fallback: moda global (ruas sem nenhum valor conhecido)
faltam_apos_rua = int(estagios.isna().sum())
moda_global = estagios.mode().iloc[0]
sinistros['semaforo_proximo_estagios'] = estagios.fillna(moda_global).astype('Int64')

n_total = int(sinistros['estagios_imputado'].sum())
print(f'Estágios imputados: {n_total} ({100*n_total/len(sinistros):.1f}%)')
print(f'  - pela moda da rua:    {n_total - faltam_apos_rua}')
print(f'  - pela moda global:    {faltam_apos_rua}')
print('Distribuição após imputação:', dict(sinistros['semaforo_proximo_estagios'].value_counts(dropna=False)))

Estágios imputados: 10609 (8.1%)
  - pela moda da rua:    6159
  - pela moda global:    4450
Distribuição após imputação: {np.int64(2): np.int64(84117), np.int64(3): np.int64(46061), np.int64(4): np.int64(645)}


**Tabela de imputação / tratamento de ausentes (Etapa 2):**

| Coluna | % ausente | Estratégia | Justificativa |
| --- | --- | --- | --- |
| `NUMERO` | ~50% | Remoção da coluna | Muito ausente e pouco preditiva; a localização já vem de lat/lon |
| `MODO_CONTROLE` / `semaforo_proximo_modo` | ~11,5% | Imputação por **constante** (`"Desconhecido"`) | Preserva as linhas; a ausência vira uma categoria própria |
| `semaforo_proximo_estagios` | ~8% | **Moda por rua** (`LOG1`) + *fallback* moda global | A mesma via tende a ter semáforos de configuração parecida |
| `HORA` | ~0% | Constante (meia-noite) | Guarda defensiva; quase não ocorre |
| `SEVERIDADE` (`"Não informado"`) | ~7% | Mantido como categoria; **excluído do alvo** | Não inventar gravidade; usado apenas fora da modelagem |

### 2.7 Tratamento de outliers

Valores extremos de `dist_semaforo_m` (acidentes longe da malha semafórica) são **reais**, não erros.
É possível detectá-los pela regra do IQR, **preferi manter** e crio uma versão winsorizada
(`dist_semaforo_m_cap`, limitada ao percentil 99) para modelos sensíveis a escala.

In [95]:
# Detecção de outliers em dist_semaforo_m (regra do IQR)
q1, q3 = sinistros['dist_semaforo_m'].quantile([0.25, 0.75])
limite_sup = q3 + 1.5 * (q3 - q1)
n_out = int((sinistros['dist_semaforo_m'] > limite_sup).sum())
print(f'Limite superior (IQR): {limite_sup:.0f} m | acima dele: {n_out} ({100*n_out/len(sinistros):.1f}%)')
print(sinistros['dist_semaforo_m'].describe(percentiles=[.5, .9, .95, .99]).round(1))

# Decisão: MANTER (são distâncias reais). Versão winsorizada (cap no p99) para uso opcional na modelagem.
p99 = sinistros['dist_semaforo_m'].quantile(0.99)
sinistros['dist_semaforo_m_cap'] = sinistros['dist_semaforo_m'].clip(upper=p99)
print(f'Coluna winsorizada criada: dist_semaforo_m_cap (cap em p99 = {p99:.0f} m)')

Limite superior (IQR): 742 m | acima dele: 8971 (6.9%)
count    130823.0
mean        268.9
std         402.7
min           0.0
50%         147.0
90%         609.5
95%         851.4
99%        2087.7
max        6675.3
Name: dist_semaforo_m, dtype: float64
Coluna winsorizada criada: dist_semaforo_m_cap (cap em p99 = 2088 m)


### 2.8 Dataset final limpo

Removi a geometria (mantendo `LATITUDE`/`LONGITUDE`) e salvei o resultado integrado e limpo em CSV.

In [96]:
dataset_final = pd.DataFrame(sinistros.drop(columns='geometry'))
dataset_final.to_csv('data/sinistros_semaforos_integrado.csv', index=False)
print('Dataset final salvo em: data/sinistros_semaforos_integrado.csv')
print('Dimensões (linhas x colunas):', dataset_final.shape)
dataset_final.sample(5)

Dataset final salvo em: data/sinistros_semaforos_integrado.csv
Dimensões (linhas x colunas): (130823, 37)


,COD_ACIDENTE,DIA,MES,ANO,LOG1,LOG2,LATITUDE,LONGITUDE,TIPOCRUZAMENTO,CONTROLETRAFEGO,...,densidade_semaforos_100m,densidade_semaforos_250m,semaforo_proximo_codigo,semaforo_proximo_status,semaforo_proximo_modo,semaforo_proximo_estagios,dist_semaforo_m,tem_semaforo_interseccao,estagios_imputado,dist_semaforo_m_cap
73931,350873,25,10,2019,"AVENIDA DOMINGOS OLIMPIO,","RUA DOM JERONIMO D06,",-3.733201,-38.543457,Cruz,Pare,...,0,4,887,CENTRALIZADO,ECOTRAFIX CONJUGADO,3,121.705325,0,False,121.705325
56194,336380,26,10,2018,"AVENIDA CEL CARVALHO, 1261",",",-3.713319,-38.588960,Meio de Quadra,Não informado,...,1,1,479,CONVENCIONAL,LOCAL,3,56.019924,0,False,56.019924
49792,314497,31,10,2017,"AVENIDA WASHINGTON SOARES, 9176",",",-3.831760,-38.482424,Meio de Quadra,Não informado,...,0,1,DETRAN,DETRAN,Desconhecido,2,228.211553,0,True,228.211553
124661,VDJ0992,12,7,2024,Luiz Vieira (Perimetral),Não informado,-3.790361,-38.587999,Meio de Quadra,Não informado,...,2,3,429,CENTRALIZADO,ECOTRAFIX,3,12.362189,1,False,12.362189
122054,WDD5416,29,4,2024,Avenida Governador Raul Barbosa,Não informado,-3.765766,-38.508899,Meio de Quadra,Não Há Controle,...,0,0,880,CONVENCIONAL,LOCAL CONJUGADO,2,780.292083,0,False,780.292083


---
## 3. Análise Exploratória e Consultas SQL

Nesta parte preparo os dados em formato *tidy*, exporto em **Parquet**, faço **consultas SQL**
(com DuckDB) e a **análise exploratória** (univariada, bivariada e multivariada), terminando com
**testes de hipóteses** ligados às perguntas de pesquisa (P1/P2) e às hipóteses H1 e H2.

In [97]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Uso o dataset já em memória (Parte 2); se não existir, recarrego do CSV.
try:
    dados = dataset_final.copy()
except NameError:
    dados = pd.read_csv('data/sinistros_semaforos_integrado.csv',
                        low_memory=False, parse_dates=['data_hora'])

print('Dados para análise:', dados.shape)

Dados para análise: (130823, 37)


### 3.1 Formato Tidy e exportação em Parquet

O dataset já está em formato *tidy*: cada linha é um sinistro (uma observação) e cada coluna é uma
variável. Aqui apenas **padronizo os tipos** (categóricas e data), mostro um exemplo de
reestruturação *wide → long* (contagem de veículos) e **exporto em Parquet**, que preserva os tipos.

In [98]:
# Padronização de tipos (cada variável com o tipo adequado)
categoricas = ['TIPOCRUZAMENTO', 'CONTROLETRAFEGO', 'USOSOLO', 'NATUREZA', 'SEVERIDADE',
               'semaforo_proximo_status', 'semaforo_proximo_modo']
for coluna in categoricas:
    dados[coluna] = dados[coluna].astype('category')
dados['data_hora'] = pd.to_datetime(dados['data_hora'], errors='coerce')

# Exemplo de reestruturação wide -> long: uma linha por (sinistro, tipo de veículo) com qtde > 0
veiculos = ['AUTOMOVEL', 'ONIBUS', 'BICICLETA', 'CAMINHAO', 'MICROONIBUS',
            'CICLOMOTOR', 'MOTOCICLETA', 'TRACAOANIMAL', 'TREM']
sinistros_veiculos_long = dados.melt(id_vars='COD_ACIDENTE', value_vars=veiculos,
                                     var_name='tipo_veiculo', value_name='quantidade')
sinistros_veiculos_long = sinistros_veiculos_long[sinistros_veiculos_long['quantidade'] > 0]
print('Tabela long de veículos (exemplo de tidy):', sinistros_veiculos_long.shape)

# Exportação em Parquet via DuckDB (escritor nativo; evita o conflito pandas 2.3 x pyarrow 21)
CAMINHO_PARQUET = 'data/sinistros_semaforos_integrado.parquet'
duckdb.sql(f"COPY (SELECT * FROM dados) TO '{CAMINHO_PARQUET}' (FORMAT PARQUET)")
print('Parquet salvo:', CAMINHO_PARQUET)

sinistros_veiculos_long.sample(5)

Tabela long de veículos (exemplo de tidy): (176081, 3)
Parquet salvo: data/sinistros_semaforos_integrado.parquet


,COD_ACIDENTE,tipo_veiculo,quantidade
111398,XZF6740,AUTOMOVEL,1
911258,JHK6553,MOTOCICLETA,1
60429,315201,AUTOMOVEL,2
820228,295735,MOTOCICLETA,1
31617,302130,AUTOMOVEL,1


### 3.2 Consultas SQL (DuckDB)

Registro o DataFrame como uma tabela SQL chamada `dados` e faço 6 consultas: tendência temporal,
comparações entre grupos (semáforo na interseção, modo de controle, nº de estágios), um *ranking*
com função de janela e uma consulta com **CTE**. A coluna "graves" agrega `Fatal` + `Ferido`
(excluindo `Não informado`).

In [99]:
con = duckdb.connect()        # banco em memória
con.register('dados', dados)  # registra o DataFrame como a tabela SQL "dados"
print('Tabela "dados" registrada no DuckDB.')

Tabela "dados" registrada no DuckDB.


In [100]:
# Q1 — Tendência temporal: total de sinistros e % de graves por ano
con.sql("""
    SELECT ANO,
           COUNT(*) AS total_sinistros,
           ROUND(100.0 * AVG(CASE WHEN SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END), 1) AS pct_graves
    FROM dados
    WHERE SEVERIDADE <> 'Não informado'
    GROUP BY ANO
    ORDER BY ANO
""").df()

,ANO,total_sinistros,pct_graves
0,2015,13960,53.1
1,2016,17450,57.3
2,2017,15452,65.7
3,2018,10848,80.4
4,2019,11091,90.8
5,2020,8621,93.2
6,2021,10162,95.3
7,2022,10546,96.1
8,2023,11127,96.6
9,2024,12324,93.8


In [101]:
# Q2 — Comparação entre grupos: taxa de graves COM vs SEM semáforo na interseção (<= 30 m)
con.sql("""
    SELECT CASE WHEN tem_semaforo_interseccao = 1 THEN 'Com semáforo (<=30m)'
                ELSE 'Sem semáforo' END AS grupo,
           COUNT(*) AS total,
           ROUND(100.0 * AVG(CASE WHEN SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END), 1) AS pct_graves
    FROM dados
    WHERE SEVERIDADE <> 'Não informado'
    GROUP BY grupo
    ORDER BY grupo
""").df()

,grupo,total,pct_graves
0,Com semáforo (<=30m),22827,72.0
1,Sem semáforo,98754,81.1


In [102]:
# Q3 — Taxa de graves por modo de controle do semáforo mais próximo (H2: SCOOT vs demais)
con.sql("""
    SELECT semaforo_proximo_modo AS modo,
           COUNT(*) AS total,
           ROUND(100.0 * AVG(CASE WHEN SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END), 1) AS pct_graves
    FROM dados
    WHERE SEVERIDADE <> 'Não informado' AND semaforo_proximo_modo IS NOT NULL
    GROUP BY modo
    ORDER BY total DESC
""").df()

,modo,total,pct_graves
0,LOCAL,59284,85.3
1,ECOTRAFIX,15518,77.5
2,SCOOT,14903,66.4
3,Desconhecido,9519,74.9
4,SCOOT CONJUGADO,8968,68.3
5,ECOTRAFIX CONJUGADO,7915,78.7
6,LOCAL CONJUGADO,5474,83.4


In [103]:
# Q4 — Taxa de graves por nº de estágios (apenas valores CONHECIDOS, não imputados) — H1
con.sql("""
    SELECT semaforo_proximo_estagios AS estagios,
           COUNT(*) AS total,
           ROUND(100.0 * AVG(CASE WHEN SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END), 1) AS pct_graves
    FROM dados
    WHERE SEVERIDADE <> 'Não informado' AND NOT estagios_imputado
    GROUP BY estagios
    ORDER BY estagios
""").df()

,estagios,total,pct_graves
0,2,72351,78.8
1,3,39053,81.7
2,4,602,70.3


In [104]:
# Q5 — Ranking das naturezas de sinistro por frequência (função de janela RANK)
con.sql("""
    SELECT NATUREZA,
           COUNT(*) AS total,
           RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
    FROM dados
    GROUP BY NATUREZA
    ORDER BY ranking
    LIMIT 10
""").df()

,NATUREZA,total,ranking
0,Colisão,58544,1
1,Queda,15444,2
2,Colisão Transversal,14078,3
3,Colisão Lateral,12942,4
4,Atropelamento,11180,5
5,Colisão Traseira,9368,6
6,Choque c/ Obstáculo Fixo,2812,7
7,Colisão Frontal,1342,8
8,Não informado,1254,9
9,Engavetamento,1180,10


In [105]:
# Q6 — CTE: compara a taxa de graves de cada modo com a taxa GERAL (diferença em pontos percentuais)
con.sql("""
    WITH base AS (
        SELECT * FROM dados WHERE SEVERIDADE <> 'Não informado'
    ),
    geral AS (
        SELECT AVG(CASE WHEN SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END) AS taxa_geral
        FROM base
    )
    SELECT b.semaforo_proximo_modo AS modo,
           COUNT(*) AS total,
           ROUND(100.0 * AVG(CASE WHEN b.SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END), 1) AS pct_graves,
           ROUND(100.0 * (AVG(CASE WHEN b.SEVERIDADE IN ('Fatal','Ferido') THEN 1.0 ELSE 0 END) - g.taxa_geral), 1) AS diff_vs_geral_pp
    FROM base b CROSS JOIN geral g
    WHERE b.semaforo_proximo_modo IS NOT NULL
    GROUP BY b.semaforo_proximo_modo, g.taxa_geral
    ORDER BY total DESC
""").df()

,modo,total,pct_graves,diff_vs_geral_pp
0,LOCAL,59284,85.3,5.9
1,ECOTRAFIX,15518,77.5,-1.9
2,SCOOT,14903,66.4,-13.0
3,Desconhecido,9519,74.9,-4.5
4,SCOOT CONJUGADO,8968,68.3,-11.1
5,ECOTRAFIX CONJUGADO,7915,78.7,-0.7
6,LOCAL CONJUGADO,5474,83.4,4.0


**Interpretação das consultas:**

- **Q1 (temporal):** o % de graves "sobe" de ~53% (2015) para ~96% (2023). Isso reflete sobretudo a
  **queda no registro de "Ileso"** ao longo dos anos (mudança de critério), e não necessariamente
  uma piora real — por isso `ANO` deve ser usado com cautela.
- **Q2 (semáforo na interseção):** sinistros a ≤ 30 m de um semáforo têm **menor % de graves
  (72,0%) do que os demais (81,1%)** — coerente com a ideia de que a sinalização organiza o fluxo.
- **Q3 / Q6 (modo de controle):** o modo **SCOOT** (adaptativo) tem a **menor** taxa de graves
  (~66%), cerca de 13 p.p. abaixo da média geral; o modo **LOCAL** tem a **maior** (~85%).
- **Q4 (estágios):** a relação **não é monotônica** — 3 estágios tem a maior taxa (81,7%) e 4 a
  menor (70,3%, mas com apenas 602 casos). Ou seja, "mais estágios = menos graves" (H1) **não se
  confirma de forma simples**.
- **Q5 (ranking):** "Colisão" é de longe a natureza mais frequente, seguida de "Queda" e de outros
  tipos de colisão.

Essas diferenças são testadas estatisticamente na seção 3.4.

### 3.3 Análise Exploratória

Análise **univariada** (distribuições), **bivariada** (relações com a gravidade) e **multivariada**
(matriz de correlação).

In [ ]:
# Univariada: distribuição da gravidade e da distância ao semáforo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dados['SEVERIDADE'].value_counts().plot(kind='bar', ax=axes[0], title='Distribuição de SEVERIDADE')
axes[0].set_ylabel('nº de sinistros')
dados['dist_semaforo_m'].clip(upper=1000).plot(kind='hist', bins=40, ax=axes[1],
        title='Distância ao semáforo mais próximo (m, limitado a 1000)')
axes[1].set_xlabel('metros')
plt.tight_layout(); plt.show()

dados[['dist_semaforo_m', 'densidade_semaforos_100m', 'densidade_semaforos_250m']].describe().round(1)

In [ ]:
# Bivariada: taxa de graves por presença de semáforo e por nº de estágios
dados_rotulados = dados[dados['SEVERIDADE'] != 'Não informado'].copy()
dados_rotulados['grave'] = dados_rotulados['SEVERIDADE'].isin(['Fatal', 'Ferido']).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(dados_rotulados.groupby('tem_semaforo_interseccao', observed=True)['grave'].mean() * 100).plot(
    kind='bar', ax=axes[0], title='% de graves por semáforo na interseção')
axes[0].set_xticklabels(['Sem (>30m)', 'Com (<=30m)'], rotation=0); axes[0].set_ylabel('% graves')
(dados_rotulados.groupby('semaforo_proximo_estagios', observed=True)['grave'].mean() * 100).plot(
    kind='bar', ax=axes[1], title='% de graves por nº de estágios')
axes[1].set_ylabel('% graves'); axes[1].set_xlabel('estágios')
plt.tight_layout(); plt.show()

In [ ]:
# Multivariada: matriz de correlação das variáveis numéricas
num_cols = ['dist_semaforo_m', 'densidade_semaforos_100m', 'densidade_semaforos_250m',
            'semaforo_proximo_estagios', 'MORTOS', 'FERIDOS', 'ILESOS']
correlacao = dados[num_cols].corr(numeric_only=True)
plt.figure(figsize=(7, 6))
sns.heatmap(correlacao, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlação (variáveis numéricas)')
plt.tight_layout(); plt.show()

### 3.4 Teste de hipóteses

Uso o **teste qui-quadrado de independência** para verificar, de forma preliminar, se a gravidade
(graves = Fatal/Ferido vs. Ileso) está associada a:
- **H1** — o número de estágios do semáforo mais próximo;
- **H2** — o modo de controle ser adaptativo (SCOOT) ou não.

Um p-valor pequeno (< 0,05) indica associação estatisticamente significativa.

In [ ]:
# H1: nº de estágios x gravidade (apenas valores CONHECIDOS, não imputados)
conhecidos = dados_rotulados[~dados_rotulados['estagios_imputado'].astype(bool)]
tabela_h1 = pd.crosstab(conhecidos['semaforo_proximo_estagios'], conhecidos['grave'])
chi2_h1, p_h1, _, _ = chi2_contingency(tabela_h1)
print(f'H1 (estágios x gravidade, só conhecidos):  qui-quadrado={chi2_h1:.1f}  |  p-valor={p_h1:.2e}')

# H2: ser SCOOT (controle adaptativo) x gravidade
dados_rotulados['eh_scoot'] = dados_rotulados['semaforo_proximo_modo'].astype(str).str.startswith('SCOOT')
tabela_h2 = pd.crosstab(dados_rotulados['eh_scoot'], dados_rotulados['grave'])
chi2_h2, p_h2, _, _ = chi2_contingency(tabela_h2)
print(f'H2 (SCOOT x gravidade):     qui-quadrado={chi2_h2:.1f}  |  p-valor={p_h2:.2e}')

print('\nTaxa de graves por estágios (conhecidos):')
print((conhecidos.groupby('semaforo_proximo_estagios', observed=True)['grave'].mean() * 100).round(1))
print('\nTaxa de graves SCOOT vs demais:')
print((dados_rotulados.groupby('eh_scoot', observed=True)['grave'].mean() * 100).round(1))

### 3.5 Insights para a modelagem

- **As variáveis semafóricas importam.** Presença de semáforo na interseção, **modo de controle**
  (SCOOT associado a menor gravidade) e densidade aparecem como candidatas fortes para o modelo.
- **H2 (SCOOT) é fortemente sustentada** pelos dados: SCOOT ~67% de graves vs ~82% nos demais
  (qui-quadrado com p ≈ 0). Já **H1 (mais estágios → menos graves) não se sustenta** de forma clara
  — há associação significativa, mas a direção não é monotônica.
- **Cuidado com `ANO`.** A queda no registro de "Ileso" infla o % de graves ao longo do tempo;
  `ANO` está confundido com o alvo e é candidato a exclusão na modelagem.
- **Desbalanceamento.** "Graves" dominam (~79% dos casos rotulados) e "Fatal" é raro (~1,5%). Vale
  usar métricas além da acurácia (AUC, recall) e tratar o desbalanceamento.
- **Associação não é causa.** Os contrastes podem ter confundidores (ex.: SCOOT fica em grandes
  corredores centrais). A modelagem multivariada da Etapa 4 ajuda a isolar os efeitos.